# 🧠 AI Logic Engine: Main Inference Console

Firestore ထဲရှိ အသိပညာများကို အခြေခံ၍ အကျိုးအကြောင်း (Logical Path) ရှာဖွေပေးသည်။

In [ ]:
# Step 1: Initialize
!pip install -q firebase-admin

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore
import re

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Inference Engine Memory Ready: {DATABASE_ID}")

In [ ]:
# Step 2: Symbolic Logical Engine
class KnowledgeInferenceEngine:
    def __init__(self, db):
        self.db = db
        self.cache = {}

    def clean_id(self, text):
        if not text: return ""
        text = text.lower().strip()
        return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

    def solve_logic_chain(self, start, target, depth=10):
        sid, tid = self.clean_id(start), self.clean_id(target)
        if sid == tid: return [f"{start} is already recognized as {target}."]
        
        queue = [(sid, [])]
        visited = {sid}
        
        while queue:
            curr_id, path = queue.pop(0)
            if curr_id == tid: return path
            if len(path) >= depth: continue
            
            # Fetch Current Node Knowledge
            doc_ref = self.db.collection('nodes').document(curr_id)
            doc = doc_ref.get()
            if not doc.exists: continue
            
            data = doc.to_dict()
            
            # Path 1: Taxonomy Groups (Inheritance)
            for g in data.get('groups', []):
                if g not in visited:
                    visited.add(g)
                    queue.append((g, path + [f"{curr_id} is a type of {g}"]))
            
            # Path 2: Direct Relations
            for rel in data.get('relations', []):
                target_ptr = rel.get('targetId')
                if target_ptr and target_ptr not in visited:
                    visited.add(target_ptr)
                    explanation = rel.get('sentence') or f"{curr_id} {rel['verb']} {target_ptr}"
                    queue.append((target_ptr, path + [explanation]))
                    
        return None

engine = KnowledgeInferenceEngine(db)
print("✅ Inference Core Stabilized.")

In [ ]:
# @title 🔍 Step 3: Run Inference Query
CONCEPT_A = "Mitochondria" # @param {type:"string"}
CONCEPT_B = "Cellular Energy" # @param {type:"string"}
MAX_DEPTH = 10 # @param {type:"slider", min:1, max:20, step:1}

print(f"🧠 Calculating Logical Path: '{CONCEPT_A}' ➔ '{CONCEPT_B}'...\n")
proof = engine.solve_logic_chain(CONCEPT_A, CONCEPT_B, depth=MAX_DEPTH)

if proof:
    print(f"✅ Proof Found! Logical chain results below:\n")
    for i, step in enumerate(proof):
        print(f"  [{i+1}] {step}")
else:
    print("❌ No logical connection found within this search depth.")
    print("💡 Suggestion: Ingest more data to build more connections.")